# CSC 421 - Constraint Satisfaction Problems

### Instructor: Shengyao Lu

We have used **atomic** representation to solve path finding problems by searching in trees or graphs. We also found in informed search algorithms that <mark>domain-specific heuristics could estimate the cost</mark> of reaching the goal from a given state. 

In today's class, we will look into problems using a **factored representation** for each state, which refers to a set of variables, each of which has a value. In these problems, the structure of the states is somewhat awared, and use general rather than domain-specific heuristics. 

### Readings
- Basic: Sections 6.1, 6.2, 6.3, 6.4, and Summary
- Expected: 6.5, 4.1, 4.2
- Advanced: All the chapter including bibliographical and historical notes

## 1. Defining Constraint Satisfaction Problems (CSP)

In many real-world problems, we don't care about the best solution, even not a nearly optimial solution. Instead, we just want a "good" solution that works and everyone is happy with it.  
We can model such a problem as a Constraint Satisfaction Problem (CSP):  _if all constraints are satisfied, then we are done._

A CSP problem consists of three components, $\mathcal{X,D,C}$.
- $\mathcal{X}$ := a set of variables, $\{X_1, \dots, X_n\}$.
- $\mathcal{D}$ := a set of domains, $\{D_1, \dots, D_n\}$.
    - A domain $D_i$ consists of a set of allowable values $\{v_1\dots v_k\}$ for variable $X_i$. $k$ := dimension of $D_i$, different variables can have different domains of different sizes. 
    - e.g., a boolean variable $X_{bool}$ have the domain $\{true, false\}$
- $\mathcal{C}$ := a set of constraints that specify allowable combinations of values.
    - Each constraint $C_j$ consists of a pair $\left\langle scope,rel \right\rangle$.
        - $scope$ := a tuple of variables in $C_j$.
        - $rel$ := a relation that defines the values that $scope$ can take on. 
    - e.g., if $X_1,X_2$ both have the domain $\{1,2,3\}$, then two ways to represent the constraint $C_m$ are, where $C_m$ := $X_1$ must be greater than $X_2$:
        - $\left\langle (X_1,X_2), \{(3,1), (3,2), (2,1)\}\right\rangle$
        - $\left\langle (X_1,X_2),X_1>X_2\right\rangle$

### Terminology 

- **Assignment of values to variables:** e.g. $\{X_i=v_i, X_j=v_j,\dots\}$. 
- **Complete assignment:** each variable is assigned a value.  
- **Partial assignment:** leaves some variables unassigned.
- **Consistent (legal) assignment:** Not violate any constraints.
- **Solution to a CSP:** a consistent, complete assignment. 
- **Partial solution:** a consistent partial assignment.


### Example problem: Midterm schedule

Now consider we are going to have **our midterm for this course on Thursday, Feb 26**, at the Computer-Based Testing Lab (CBTL). There will be 4 time slots for the students to choose. You may choose whichever time slot that works for you best. However, we are not the only course holding a midterm on February 26. Other courses may include OS (Operating System), DB (database), ALG (Algorithms), MATH, and we are the AI course. 

Now let's **Assume** the following:
- the four time slots are:
    - T1: 8:30 A.M. - 9:15 A.M.
    - T2: 9:30 A.M. - 10:15 A.M.
    - T3: 12:30 P.M. - 1:15 P.M.
    - T4: 2:00 P.M. - 2:45 P.M.
- Each time slot allows at most two students to take the midterm for a given course.
- There are 4 students we need to allocate. None of them have other things to do on Feb 26. 
    - A: OS, AI
    - B: DB, MATH
    - C: AI, ALG
    - D: AI, DB

Then, how to formalize a CSP in this case? i.e., what are the three components, $\mathcal{X,D,C}$?

1. $\mathcal{X}$: Consider ${X}_{s,c}=\text{student } s \text{ takes course } c \text{ in which time slot}$
- $X_{A,OS},X_{A.AI}$
- $X_{B,DB}, X_{B,MATH}$
- $X_{C,AI}, X_{C,ALG}$
- $X_{D,AI}, X_{D,DB}$

    Therefore, $\mathcal{X}=\{X_{A,OS},X_{A.AI}, X_{B,DB}, X_{B,MATH}, X_{C,AI}, X_{C,ALG}, X_{D,AI}, X_{D,DB}\}$

2. $\mathcal{D}$: four time slots.   
$D(\cdot)\in \mathcal{D}$   
$D(X_{s,c})=\{T1,T2,T3,T4\}$

3. $\mathcal{C}$: Two main constraints:
- $C_1$: A student cannot take two different course exams in the same time slot.
    - four binary constraints
        - $\langle \{X_{A,OS},X_{A,AI}\}, X_{A,OS} \neq X_{A.AI}\rangle$
        - $\langle \{X_{B,DB}, X_{B,MATH}\} , X_{B,DB} \neq X_{B,MATH}\rangle$
        - $\langle \{X_{C,AI}, X_{C,ALG}\} , X_{C,AI} \neq X_{C,ALG}\rangle$
        - $\langle \{X_{D,AI}, X_{D,DB}\} , X_{D,AI} \neq X_{D,DB}\rangle$
- $C_2$: At most two students can be scheduled in the same time slot.
    - scope: $\mathcal{X}$
    - rel: $\forall t\in \{T1,T2,T3,T4\}, \sum_{X \in \mathcal{X}} {\mathbf{1}[X=t]\leq 2}$

### Example Problem: map coloring 

<img src="images/csp_australia.png" width="800px">

Looking at a map of Australia showing each of its states and territories. The task is "coloring each region either **red, green, or blue** in such a way that **no two neighboring regions have the same color.**" 
* $\mathcal{X}=\{{WA}, {NT}, Q, {NSW}, V, {SA}, T\}$
* $\forall D_i \in \mathcal{D}, D_i=\{red, green,blue\}$
* $\mathcal{C}=\{{SA} \neq WA, SA\neq NT, SA\neq Q, SA\neq NSW, SA\neq V, WA\neq NT, NT\neq Q, Q\neq NSW, NSW\neq V\}$
    * where ${SA} \neq WA$ is a shortcut for $\langle (SA,WA),SA\neq WA\rangle$. It can be fully enumerated in turn as    
        $\{(red, green), (red, blue), (green, red), (green,blue), (blue, red), (blue,green)\}$.

There are many possible solutions, including:  
$\{WA=red, NT=green, Q=red, NSW=green, V=red, SA=blue, T=red\}$

(b) in the figure is a **constraint graph**, where the nodes correspond to variables of the problem, an edge connecting any two nodes means the two variables participate in a constraint. 

### Variations on CSP formalism
Domains:
1. discrete vs. continuous domains 
2. finite vs. infinite domains  

Constrains:
1. unary vs. binary, higher-order
    * unary: $\langle SA, SA\neq red\rangle$
    * binary: $\langle (SA,WA),SA\neq WA\rangle$
2. hard (absolute) vs. soft (preference)
    * hard: must be satisfied
    * soft: express some notion of solutions are preferred over others.


Terms: 
* Simplest: **discrete, finite domains** (e.g. midterm schedule, map coloring)
* **discrete, infinite domains:** e.g., no deadline to attend the midterms. In this case, we may give **linear constraints** like $T1+45 mins \leq T2$. 
* **continuous-domain CSP**: e.g., linear programming problems
* **binary CSP:** with only unary and binary constraints 
* **global constraint:** a constraint with an arbitrary number of variables.
* **constraint hypergraph:** the constraint graph with both ordinary nodes, and hypernodes reprensenting auxiliary variables.  

## 2. Consistency

**Node consistency:** For a single variable, we say it is node-consistent when all the values in its domain satisfy its unary constraints. 
- $D_A=\{Mon,Tue\}$, $C_{A}=\{A\neq Fri\}$  

**Arc consistency:** A variable in CSP is said to be arc-consistent, when every value in its domain satisfies its binary constraints. 
- $A$ is arc-consistent w.r.t. $B$: $\forall v \in D_A,\ \exists u \in D_B \text{ such that } C_{A,B}(v,u) \text{ holds.}$ 
    - e.g., $D_A=\{Mon,Tue\}, D_B=\{Fri, Sat\}$, $C_{A,B}=\{A\neq B\}$.
- A graph is arc-consistent if every variable is arc-consistent with every other variable.   

### Enforce Arc consistency
- AC-3 does not search for an assignment. Instead, it enforces arc consistency by removing domain values that violate binary constraints. If any variable’s domain becomes empty, the CSP is inconsistent.
- To make X arc-consistent with respect to Y, remove elements from X's domain until every choice for X has a possible choice for Y.


<img src="images/arc-consist-pseudo.png" width="600px">

(See example)

## 3. Backtracking Search

Recall, for traditional search problems:
* initial state
* actions
* transition model
* goal test
* path-cost function

For CSPs:
* initial state: empty assignment
* actions: add a $\{variable=value\}$ to assignment
* transition model: shows how perform an action changes the current assignment
* goal test: check if all variables assigned and constraints all satisfied
* path-cost function: all paths have same cost


### Pseudocode of Backtracking Search

<img src="images/backtrack-pseudo.png" width="600px">

(see example x2)

In [ ]:
# Recursive backtracking - start with an initial assignment (usually empty) and then select a variable and 
# assign it a value. If the current assignment is consistent with the constraints call recursively 

def recursive_backtracking(assignment, csp):
    if isComplete(assignment):
        return assignment
    var = select_unassigned_variable(csp["VARIABLES"], assignment)
    for value in csp["DOMAINS"]:
        assignment[var] = value
        if is_consistent(assignment, csp["CONSTRAINTS"]):
            result = recursive_backtracking(assignment, csp)
            if result != "FAILURE":
                return result
        assignment[var] = None
    return "FAILURE"

### Inference during backtracking search
- **Maintaining arc-consistency:** Although backtracking search is more efficient than simple search, it still takes a lot of computational power. Enforcing arc consistency, on the other hand, is less resource intensive. By interleaving backtracking search with inference (enforcing arc consistency), we can get at a more efficient algorithm. 
    - When we make a new assignment to X, calls AC-3, starting with a queue of all arcs (Y, X) where Y is a neighbor of X.

(see example)

### Variable and value ordering

There are additional ways to make the algorithm more efficient. So far, we selected an unassigned variable randomly. However, some choices are more likely to bring to a solution faster than others. This requires the use of **heuristics**.

<img src="images/recursive_backtracking_csp_australia.png" width="800px">

### Heuristics to use during backtracking search 

1. **Minimum remaining values (MRV):**  
    - choose the variable with the fewest legal values left.
    - If a variable’s domain was constricted by inference, and now it has only one value left (or even if it’s two values), then by making this assignment we will reduce the number of backtracks we might need to do later. 
    - That is, we will have to make this assignment sooner or later, since it’s inferred from enforcing arc-consistency. If this assignment brings to failure, it is better to find out about it as soon as possible and not backtrack later.
2. **Degree heuristic:** 
    - Relies on the degrees of variables, where a degree is how many arcs connect a variable to other variables. 
    - By choosing the variable with the highest degree, with one assignment, we constrain multiple other variables, speeding the algorithm’s process.
3. **Least constraining variable (LCV):** 
    - select the value that will constrain the least other variables.
    - That is, we want to locate what could be the largest potential source of trouble (the variable with the highest degree), and then render it the least troublesome that we can (assign the least constraining value to it).

<img src="images/least_constraining_variable_csp.png" width="900px">

4. **Forward Checking:** keep track of remaining legal values for unassigned variables 

<img src="images/forward_checking_csp.png" width="900px">

<img src="images/forward-checking-csp.png" width="900px">

(see example x2)

## 4. Summary 


1. CSPs are a special kind of problem 
2. States defined by values of a fixed set of variables
3. Goal test defined by constraints of variable values
4. Back-tracking = depth-first search with one variable assigned per node
5. Variable ordering and value selection heuristics can help significantly
6. Forward checking prevents assignments that guarantee later failure 
7. Specific-constraint type and structure (for example trees) can lead to more efficient solvers 

## 5. Exercise 
### ❓Problem: Graph Coloring as a CSP

Consider the following constraint satisfaction problem.
- Each node in the graph is a variable: $\mathcal{X}=\{A,B,C,D,E,F\}$
- Each variable has the same initial domain: $D(X)=\{r,g,b\}$
- For every edge $(X,Y)$ in the graph: $X\neq Y$.
- Edges: $(A,B), (A,C), (B,C), (B,D), (C,E), (D,E), (E,F)$.

You must solve this CSP using Backtracking Search with:
- MRV and Degree heuristic for variable selection
- LCV for value ordering
- Forward Checking after each assignment

Assume the following tie-breaking rules:
- If MRV ties, break ties using the degree heuristic first, then alphabetical.
- If LCV ties, break ties in the order of $r,g,b$

<img src="images/csp-ex.png" width="600px">

Q: What is the first complete assignment found by the algorithm? 


## 6. Local Search

Local search algorithms are very effective in solving many CSPs. They use a complete-state formulation where each state assigns a value to every variable, and the search changes the value of one variable at a time.

### Hill climbing
- It is a local search algorithm, also called greedy local search. 
- the idea is to find a **high-scoring state by searching from a start state to neighboring states**, without keeping track of the paths, nor the set of states that have been reached.
- Pros:
    - keeps only a small number of states in memory.
    - they can often find reasonable solutions in **large or infinite state spaces** for which systematic algorithms are unsuitable.
- Cons:
    - easily get stuck in a local maximum.

<img src="images/hill-climb-psuedo.png" width="600px">
<br></br>
(see bar example)


| Variant           | Definition                                   | Pros                                    | Cons |
|-------------------|------------------|-----------------------------------------|------------------------------|
| steepest-ascent   | choose the best-valued neighbor              | fast, simple                            | easily gets stuck at local maxima or plateaus |
| stochastic        | choose randomly from better-valued neighbors | may escape local maxima                 | inefficient                                   |
| first-choice      | choose the first higher-valued neighbor      | low overhead; works well with large $b$ | depends on neighbor ordering                  |
| random-restart    | conduct hill climbing multiple times         | higher chance to find global optimum    | costly                                        |
| local beam search | chooses the k highest-valued neighbors       | shares information among states         | costly                                        |



### Simulated annealing
- It is a local search algorithm, a version of stochastic hill climbing.
- Early on, higher "temperature": **more** likely to accept neighbors that are worse than current state.
- Later on, lower "temperature": **less** likely to accept neighbors that are worse than current state.

<img src="images/sim-anneal-pseudo.png" width="600px">


### CSP example
I am the mayor of a small town. There are **4 households** in the town. For the well-being of the town, I want to build **2 hospitals**, and construct roads connecting the hospitals to the four households.
- My funding is limited. After paying for the hospitals, I only have enough money to build **11 grid segments of roads**.
- Assume that each time a road crosses **one grid cell edge**, it costs **1 unit**.
- For aesthetic reasons, I am **not allowed to build diagonal roads** (i.e., road length is measured using Manhattan distance).
- Each hosipital can take care of **at most 2 households**. 
- (People in this town are shy. They don't want to meet each other when go to the hospital. (They shouldn't share the road))

❓ Where should I place the two hospitals so that all households can be connected within the funding limit?   
Please formalize this problem as a CSP, and then solve it using the hill-climbing algorithm we just learned (use the basic hill-climbing variant).

<img src="images/hill-climb-ex.png" width="600px">

- Variables 
    - $\mathcal{X}=\{X_1, X_2\}:$ the grid cells chosen for Hospital 1 and Hospital 2.
    - $\mathcal{A}=\{A_1,A_2,A_3,A_4\}:$ which house is allocated to which hospital. 
- Domains $\mathcal{D}$
    - $\{D_{X_1}, D_{X_2}\}:$ all valid grid cells of hospitals, excluding 4 households cells.
    - $D_{A_i}\in\{1,2\}$.
- Constraints $\mathcal{C}=\{C_1, C_2, C_3\}$
    - $C_1$ Hospitals cannot be in the same cell: $X_1\neq X_2$.
    - $C_2$ capacity constraints: $\sum_{i=1}^{4} {\mathbf{1}[A_i=1]\leq 2}, \sum_{i=1}^{4} {\mathbf{1}[A_i=2]\leq 2}$.
    - $C_3$ Total road cost within budget: $\sum_{i=1}^{4} {d(X_{A_i}, H_i)}\leq 11$.
        - Each household connects to its nearest hospital (by Manhattan distance). Define Manhattan distance $d(\cdot,\cdot)$ then the total road cost is:  
        $d(X_i,H_j):=$ manhattan distance between hospital $X_i$ and household $H_j$.

_(If we use backtrack search with heuristics, MRV, LCV, forward checking are barely helpful here. We may need to try out up to $N^2$ assignments during the backtrack, where $N=5\cdot 10:=$ number of cells in this case. i.e., 2,500 tries.)_

### Linear programming

Recall that the domains a CSP can be either discrete or continuous. We have been exploring problems discrete domains. What if a problem that this? 


<img src="images/csp-lp.png" width="600px">

#### **Linear programming**

- Minimize a cost function $c_1x_1+c_2x_2+c_nx_n$.
- with constraints of form $a_1x_1 + a_2x_2 + ... + a_nx_n \leq b$, or of form $a_1x_1 + a_2x_2 + ... + a_nx_n = b$.
- With bounds for each variable $l_i \leq x_i \leq u_i$.

#### **Example** 

- Two machines $X_1$ and $X_2$. $X_1$ costs \$50/hour to run, $X_2$ costs \$80/hour to run. Goal is to minimize cost.
- $X_1$ requires 5 units of labor per hour. $X_2$ requires 2 units of labor per hour. Total of 20 units of labor to spend.
- $X_1$ produces 10 units of output per hour. $X_2$ produces 12 units of output per hour. Company needs 90 units of output.

⬇️

- Cost function: $50x_1+80x_2$
- Constraint: $5x_1+2x_2\leq 20$
- Constraint: $10x_1+12x_2\geq 90$
